# 历史分析 Notebook

已清除执行输出和本机凭据。原始数据不随仓库发布。当前可复现入口见根目录 README。可视化 Notebook 的部分查询依赖原数据库中间表，其建表与用户分层规则未完整留档；不将这些查询视为已完成复现。原手填漏斗和汇总图已移除，明确口径的重算结果见 audit_results.json。


In [ ]:
import sqlalchemy
import pymysql

print("SQLAlchemy版本：", sqlalchemy.__version__)
print("PyMySQL安装成功")

In [ ]:
import sqlalchemy
import pymysql

print("SQLAlchemy版本：", sqlalchemy.__version__)
print("PyMySQL版本：", pymysql.__version__)
print("安装成功！")

In [ ]:
# ============================================================
# 2. 连接 MySQL 数据库
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

# MySQL 配置
MYSQL_USER = "root"
MYSQL_PASSWORD = os.environ["MYSQL_PASSWORD"]
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_DATABASE = "taobao_growth"

# 创建数据库连接
engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
    f"?charset=utf8mb4"
)

print("数据库连接对象创建成功")

In [ ]:
# ============================================================
# 3. 测试数据库连接
# ============================================================

sql = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS total_users
FROM user_behavior;
"""

test_df = pd.read_sql(sql, engine)

display(test_df)

In [ ]:
# ============================================================
# 4. 每日活跃用户趋势
# ============================================================

sql_daily = """
SELECT
    behavior_date,
    COUNT(DISTINCT user_id) AS dau
FROM user_behavior
GROUP BY behavior_date
ORDER BY behavior_date;
"""

daily_df = pd.read_sql(sql_daily, engine)

daily_df["behavior_date"] = pd.to_datetime(daily_df["behavior_date"])

display(daily_df)

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = os.environ.get("CJK_FONT_PATH", "C:/Windows/Fonts/msyh.ttc")

font_manager.fontManager.addfont(font_path)

font_prop = font_manager.FontProperties(fname=font_path)
font_name = font_prop.get_name()

plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("当前中文字体：", font_name)

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    daily_df["behavior_date"],
    daily_df["dau"],
    marker="o"
)

plt.title("每日活跃用户数（DAU）变化趋势")
plt.xlabel("日期")
plt.ylabel("活跃用户数")

plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# 6. 每日各类用户行为数据
# ============================================================

sql_behavior = """
SELECT
    behavior_date,
    SUM(behavior_type = 'pv') AS pv,
    SUM(behavior_type = 'fav') AS fav,
    SUM(behavior_type = 'cart') AS cart,
    SUM(behavior_type = 'buy') AS buy
FROM user_behavior
GROUP BY behavior_date
ORDER BY behavior_date;
"""

behavior_df = pd.read_sql(sql_behavior, engine)

behavior_df["behavior_date"] = pd.to_datetime(
    behavior_df["behavior_date"]
)

display(behavior_df)

In [ ]:
# ============================================================
# 7. 每日 PV 趋势
# ============================================================

plt.figure(figsize=(12, 5))

plt.plot(
    behavior_df["behavior_date"],
    behavior_df["pv"],
    marker="o"
)

plt.title("每日浏览量（PV）变化趋势")
plt.xlabel("日期")
plt.ylabel("浏览次数")

plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# 8. 收藏 / 加购 / 购买趋势
# ============================================================

plt.figure(figsize=(12, 6))

plt.plot(
    behavior_df["behavior_date"],
    behavior_df["fav"],
    marker="o",
    label="收藏"
)

plt.plot(
    behavior_df["behavior_date"],
    behavior_df["cart"],
    marker="o",
    label="加购"
)

plt.plot(
    behavior_df["behavior_date"],
    behavior_df["buy"],
    marker="o",
    label="购买"
)

plt.title("每日收藏、加购与购买行为变化趋势")
plt.xlabel("日期")
plt.ylabel("行为次数")

plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# 11. 次日留存率趋势
# ============================================================

sql_retention_1d = """
SELECT
    a.activity_date,

    COUNT(DISTINCT a.user_id) AS active_users,

    COUNT(DISTINCT b.user_id) AS retained_users,

    ROUND(
        COUNT(DISTINCT b.user_id)
        / COUNT(DISTINCT a.user_id) * 100,
        2
    ) AS retention_rate

FROM user_daily_active a

LEFT JOIN user_daily_active b
    ON a.user_id = b.user_id
    AND b.activity_date = DATE_ADD(a.activity_date, INTERVAL 1 DAY)

WHERE a.activity_date <= '2017-12-02'

GROUP BY a.activity_date
ORDER BY a.activity_date;
"""

retention_1d = pd.read_sql(sql_retention_1d, engine)

retention_1d["activity_date"] = pd.to_datetime(
    retention_1d["activity_date"]
)

display(retention_1d)

In [ ]:
# ============================================================
# 12. 次日留存率趋势图
# ============================================================

plt.figure(figsize=(11, 5))

plt.plot(
    retention_1d["activity_date"],
    retention_1d["retention_rate"],
    marker="o"
)

plt.title("用户次日留存率变化趋势")
plt.xlabel("日期")
plt.ylabel("次日留存率（%）")

plt.ylim(0, 105)
plt.xticks(rotation=45)
plt.grid(alpha=0.3)

# 标注每个点的留存率
for x, y in zip(
    retention_1d["activity_date"],
    retention_1d["retention_rate"]
):
    plt.text(
        x,
        y + 2,
        f"{y:.1f}%",
        ha="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 13. 7日留存率
# ============================================================

sql_retention_7d = """
SELECT
    a.activity_date,

    COUNT(DISTINCT a.user_id) AS active_users,

    COUNT(DISTINCT b.user_id) AS retained_users,

    ROUND(
        COUNT(DISTINCT b.user_id)
        / COUNT(DISTINCT a.user_id) * 100,
        2
    ) AS retention_rate

FROM user_daily_active a

LEFT JOIN user_daily_active b
    ON a.user_id = b.user_id
    AND b.activity_date = DATE_ADD(a.activity_date, INTERVAL 7 DAY)

WHERE a.activity_date <= '2017-11-26'

GROUP BY a.activity_date
ORDER BY a.activity_date;
"""

retention_7d = pd.read_sql(sql_retention_7d, engine)

retention_7d["activity_date"] = pd.to_datetime(
    retention_7d["activity_date"]
)

display(retention_7d)

In [ ]:
# ============================================================
# 14. 7日留存率柱状图
# ============================================================

plt.figure(figsize=(8, 5))

bars = plt.bar(
    retention_7d["activity_date"].dt.strftime("%Y-%m-%d"),
    retention_7d["retention_rate"]
)

plt.title("用户7日留存率")
plt.xlabel("活跃队列日期")
plt.ylabel("7日留存率（%）")

plt.ylim(0, 105)

for bar, rate in zip(
    bars,
    retention_7d["retention_rate"]
):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 1,
        f"{rate:.2f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 15. 读取最终用户分群数据
# ============================================================

sql_segments = """
SELECT
    user_segment,
    COUNT(*) AS user_count,
    ROUND(COUNT(*) / (SELECT COUNT(*) FROM final_user_segments) * 100, 2) AS user_rate,
    ROUND(AVG(pv_count), 2) AS avg_pv,
    ROUND(AVG(fav_count), 2) AS avg_fav,
    ROUND(AVG(cart_count), 2) AS avg_cart,
    ROUND(AVG(buy_count), 2) AS avg_buy,
    ROUND(AVG(active_days), 2) AS avg_active_days
FROM final_user_segments
GROUP BY user_segment
ORDER BY user_count DESC;
"""

segment_df = pd.read_sql(sql_segments, engine)

display(segment_df)

In [ ]:
# ============================================================
# 16. 用户分群人数分布
# ============================================================

segment_plot = segment_df.sort_values("user_count")

plt.figure(figsize=(11, 6))

bars = plt.barh(
    segment_plot["user_segment"],
    segment_plot["user_count"]
)

plt.title("用户价值分群人数分布")
plt.xlabel("用户数量")
plt.ylabel("用户类型")

for bar, count, rate in zip(
    bars,
    segment_plot["user_count"],
    segment_plot["user_rate"]
):
    plt.text(
        bar.get_width() + 100,
        bar.get_y() + bar.get_height()/2,
        f"{count:,}（{rate:.1f}%）",
        va="center"
    )

plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 17. 各类用户平均购买次数
# ============================================================

buy_plot = segment_df.sort_values("avg_buy")

plt.figure(figsize=(11, 6))

bars = plt.barh(
    buy_plot["user_segment"],
    buy_plot["avg_buy"]
)

plt.title("不同用户群体平均购买次数")
plt.xlabel("平均购买次数")
plt.ylabel("用户类型")

for bar, value in zip(
    bars,
    buy_plot["avg_buy"]
):
    plt.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height()/2,
        f"{value:.2f}",
        va="center"
    )

plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 18. 未购买用户潜在转化分析
# ============================================================

sql_non_buyers = """
SELECT
    user_segment,
    COUNT(*) AS user_count,
    ROUND(AVG(pv_count), 2) AS avg_pv,
    ROUND(AVG(fav_count), 2) AS avg_fav,
    ROUND(AVG(cart_count), 2) AS avg_cart,
    ROUND(AVG(active_days), 2) AS avg_active_days
FROM final_user_segments
WHERE buy_count = 0
GROUP BY user_segment;
"""

non_buyer_df = pd.read_sql(sql_non_buyers, engine)

display(non_buyer_df)

In [ ]:
# ============================================================
# 19. 未购买用户分层人数
# ============================================================

order = [
    "纯浏览用户",
    "低意向未购买用户",
    "中意向未购买用户",
    "高意向未购买用户"
]

plot_df = (
    non_buyer_df
    .set_index("user_segment")
    .reindex(order)
    .reset_index()
)

plt.figure(figsize=(10, 6))

bars = plt.bar(
    plot_df["user_segment"],
    plot_df["user_count"]
)

plt.title("未购买用户潜在转化分层")
plt.xlabel("用户类型")
plt.ylabel("用户数量")

for bar, value in zip(bars, plot_df["user_count"]):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 50,
        f"{int(value):,}",
        ha="center"
    )

plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# 20. 未购买用户平均收藏 / 加购行为
# ============================================================

x = np.arange(len(plot_df))
width = 0.35

plt.figure(figsize=(11, 6))

plt.bar(
    x - width/2,
    plot_df["avg_fav"],
    width,
    label="平均收藏次数"
)

plt.bar(
    x + width/2,
    plot_df["avg_cart"],
    width,
    label="平均加购次数"
)

plt.xticks(
    x,
    plot_df["user_segment"],
    rotation=15
)

plt.title("不同意向未购买用户的收藏与加购行为")
plt.xlabel("用户类型")
plt.ylabel("平均行为次数")

plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# 21. 高意向未购买用户 TOP 20
# ============================================================

sql_high_intent = """
SELECT
    user_id,
    pv_count,
    fav_count,
    cart_count,
    intent_count,
    active_days,
    recency_days
FROM final_user_segments
WHERE user_segment = '高意向未购买用户'
ORDER BY
    intent_count DESC,
    cart_count DESC,
    active_days DESC
LIMIT 20;
"""

high_intent_users = pd.read_sql(
    sql_high_intent,
    engine
)

display(high_intent_users)

In [ ]:
# ============================================================
# 24. 导出 Tableau 所需数据
# ============================================================

import os

output_dir = "../05_tableau"
os.makedirs(output_dir, exist_ok=True)

# 每日行为趋势
behavior_df.to_csv(
    f"{output_dir}/daily_behavior.csv",
    index=False,
    encoding="utf-8-sig"
)

# 次日留存
retention_1d.to_csv(
    f"{output_dir}/retention_1d.csv",
    index=False,
    encoding="utf-8-sig"
)

# 7日留存
retention_7d.to_csv(
    f"{output_dir}/retention_7d.csv",
    index=False,
    encoding="utf-8-sig"
)

# 用户分群
segment_df.to_csv(
    f"{output_dir}/user_segments.csv",
    index=False,
    encoding="utf-8-sig"
)

# 未购买用户分析
non_buyer_df.to_csv(
    f"{output_dir}/non_buyer_segments.csv",
    index=False,
    encoding="utf-8-sig"
)

# 高意向未购买用户
high_intent_users.to_csv(
    f"{output_dir}/high_intent_users.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tableau 数据导出完成！")